### Create dataloader table for tasks on T2 images (lesion load, brain age)

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# age
table_field = "f.21003.2.0" # pv lesion load: "f.24485.2.0"

# save datloader table
table_path = "/sc-projects/sc-proj-cc15-cn-ukbiobank/analyses/Explanation-benchmark-paper/model_training/training_files/T2_flair/T2_age_table.csv"

def creat_csv_UKB():

    # where are my BIDS
    BIDS_path = "/sc-resources/ukb/data/bulk/imaging/brain/20253/"
    BIDS_path_T1 = "/sc-resources/ukb/data/bulk/imaging/brain/20252/"

    # relevant paths from BIDS:
    betted_path = "20253_{eid}_{trial}/T2_FLAIR/T2_FLAIR_brain.nii.gz"

    # warp coefficients
    coefs_to_mni_path = "20252_{eid}_{trial}/T1/transforms/T1_to_MNI_warp_coef.nii.gz"

    # labels:
    # periventricular lesions from raw file:
    columns = ["f.eid", table_field]

    ukbb_df = pd.read_parquet("/sc-resources/ukb/data/projects/33073/ukb_data/table/ukb.parquet", columns=columns)

    # write csv header
    with open(table_path, "w") as f_out:
        f_out.write("eid,raw,coefs_to_mni,target\n")

        i = 0
        # get data paths for subjects:
        p = Path(BIDS_path)
        for sub_folder in p.iterdir():
            try:
                eid = int(sub_folder.name.split('_')[1])
                trial = sub_folder.name.split('_')[2]

            except:
                continue

            print("processing eid: " + str(eid))

            sub_betted_path = betted_path.format(eid=eid, trial=trial)
            sub_betted_path = BIDS_path + sub_betted_path

            sub_coefs_to_mni_path = coefs_to_mni_path.format(eid=eid, trial=trial)
            sub_coefs_to_mni_path = BIDS_path_T1 + sub_coefs_to_mni_path

            # get age
            try:
                lesion_load = ukbb_df.loc[ukbb_df['f.eid'] == eid][table_field].iloc[0]
            except:
                continue


            all_files_exist = os.path.exists(sub_betted_path) and os.path.exists(sub_coefs_to_mni_path)

            if (type(lesion_load) is np.float32 or type(lesion_load) is np.float64) and all_files_exist and int(trial) == 2:
                f_out.write(f"{eid},{sub_betted_path},{sub_coefs_to_mni_path},{lesion_load}\n")
                print("done")

x = creat_csv_UKB()